In [2]:
import os
import json

### Add LCT Entities to AST

In [4]:
def read_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def update_conditions(node, entities, selected_types):
    if 'raw_text' in node:
        raw_text = node['raw_text']
        for entity in entities:
            if 'text' in entity and 'type' in entity:
                entity_type = entity['type']
                entity_text = entity['text']
                if entity_type in selected_types and entity_text in raw_text:
                    if entity_type not in node:
                        node[entity_type] = []
                    if entity_text not in node[entity_type]:
                        node[entity_type].append(entity_text)
    for key in node:
        if isinstance(node[key], dict):
            update_conditions(node[key], entities, selected_types)
        elif isinstance(node[key], list):
            for item in node[key]:
                if isinstance(item, dict):
                    update_conditions(item, entities, selected_types)

def process_files(lct_p2_half_path, all_entities_path, output_dir, selected_types):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    for file_name in os.listdir(lct_p2_half_path):
        if file_name.endswith(".json"):
            nct_number = file_name.split('_')[0]
            criteria_type = file_name.split('_')[1].split(".")[0]  # 'exc' oder 'inc'
            file_path = os.path.join(lct_p2_half_path, file_name)
            data = read_json_file(file_path)
            entity_file = os.path.join(all_entities_path, f"{nct_number}.json")
            entities = read_json_file(entity_file)
            update_conditions(data, entities, selected_types)
            # Save the updated file
            output_file_name = f"{nct_number}_{criteria_type}_p3.json"
            output_path = os.path.join(output_dir, output_file_name)
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

In [5]:
lct_p2_half_path = "corpora/lct_p2_half"
all_entities_path = "corpora/all_entitys"
output_directory = "corpora/lct_p3_half_all_entities" #"corpora/lct_p3_half"

## ALL LCT entities
all_entities = ['Contraindication', 'Eq-Value', 'Severity', 'Drug', 'Observation-Name', 'Age', 'Location', 'Organism-Name', 'Encounter', 'Drug-Name', 'Ethnicity', 'Modifier', 'Condition', 'Eq-Unit', 'Eq-Temporal-Unit', 'Family-Member', 'Organism', 'Other', 'Immunization-Name', 'Polarity', 'Condition-Type', 'Immunization', 'Eq-Operator', 'Eq-Temporal-Recency', 'Procedure-Name', 'Indication', 'Exception', 'Study', 'Language', 'Coreference', 'Provider', 'Acuteness', 'Life-Stage-And-Gender', 'Procedure', 'Risk', 'Death', 'Assertion', 'Allergy-Name', 'Specimen', 'Negation', 'Code', 'Stability', 'Birth', 'Criteria-Count', 'Eq-Comparison', 'Condition-Name', 'Insurance', 'Observation', 'Allergy', 'Eq-Temporal-Period']

# Example for selected entity types
selected_entity_types = [
    'Condition',
    'Drug',
    'Observation'
]

In [6]:
process_files(lct_p2_half_path, all_entities_path, output_directory, all_entities)

#### LCT Unique Entitys

In [24]:
def get_unique_entity_types(data):
    unique_types = set()
    for entity in data['entities']:
        unique_types.add(entity['type'])
    return unique_types

def process_json_files(folder_path):
    unique_types = set()
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            file_path = os.path.join(folder_path, file_name)
            with open(file_path, 'r') as file:
                data = json.load(file)
            file_unique_types = get_unique_entity_types(data)
            unique_types.update(file_unique_types)
    return list(unique_types)

In [17]:
folder_path = 'corpora/all_entities'
unique_entity_types = process_json_files(folder_path)
print("Unique entity types across all files:")
print(unique_entity_types)